# Lecture 9: Final Project Experimental Best Practices for Geometric Deep Learning

**Course context.** In geometric deep learning, we do not add symmetry priors (equivariance/invariance) as decoration. We add them to encode a scientific assumption about how the world and the task transform.

A strong final project is therefore **not** defined by a single best metric on a leaderboard. It is defined by a **clear claim**, a **fair test**, and **evidence that matches the claim**.

> **Key question:** *What benefit is the symmetry prior supposed to provide, and under what regime should that benefit appear?*

Throughout this notebook, keep one principle in mind:

- A clean, well-controlled experiment is better than a large, messy benchmark.
- Equivariant models are not guaranteed to win everywhere; they should win in regimes where the inductive bias matches the data-generating structure.

## 1) What makes an experimental claim credible?

A credible claim has four parts:

1. **Hypothesis**: a concrete statement connecting symmetry to expected behavior.
   - Example: "SE(3)-equivariance should improve sample efficiency for molecular force prediction under random rigid transforms."
2. **Controlled comparison**: architecture differences are isolated from unrelated advantages.
3. **Targeted regime**: evaluate where the claimed benefit should appear (OOD rotations, low-data, reduced augmentation, etc.).
4. **Uncertainty-aware reporting**: multiple seeds, variance, and honest effect sizes.

In this course, we care about the chain:

\[
\text{Symmetry assumption} \rightarrow \text{architectural constraint} \rightarrow \text{learning behavior} \rightarrow \text{measured outcome}.
\]

If your results skip a link in this chain, your scientific story is incomplete.

## 2) Fair baselines: compare inductive bias, not hidden advantages

A weak baseline cannot validate an equivariant method. Your baseline should be competitive and honestly tuned.

### Match the following whenever possible

- **Parameter count** (or report clearly when exact matching is impossible).
- **Optimizer** (e.g., AdamW for both models).
- **Learning-rate schedule** (same schedule family and tuning budget).
- **Batch size**.
- **Number of epochs / update steps**.
- **Early stopping policy**.
- **Data augmentation policy**.
- **Accessible input information** (no privileged features for one model only).

### Why this matters

If an equivariant model wins only because it has more parameters, stronger tuning, longer training, or extra information, you have not shown a symmetry benefit. You have shown an unfair setup.

### Practical reporting tip

Include a compact comparison table in your report:

| Model | Params | Optimizer | LR schedule | Epochs | Augmentations | Extra inputs |
|---|---:|---|---|---:|---|---|
| Baseline CNN | ... | ... | ... | ... | ... | ... |
| Rotation-equivariant CNN | ... | ... | ... | ... | ... | ... |

In [ ]:
# 1) Fair baseline comparison via parameter matching (PyTorch)
import torch
import torch.nn as nn

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

class ToyMLP(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(32, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 10),
        )
    def forward(self, x):
        return self.net(x)

class ToyEquivariantStyle(nn.Module):
    """Toy stand-in: uses a tied intermediate representation."""
    def __init__(self, hidden):
        super().__init__()
        self.lift = nn.Linear(32, hidden, bias=False)
        self.act = nn.Tanh()
        self.readout = nn.Linear(hidden, 10)
    def forward(self, x):
        return self.readout(self.act(self.lift(x)))

baseline = ToyMLP(hidden=48)
equivariant = ToyEquivariantStyle(hidden=52)

p_base = count_trainable_params(baseline)
p_eq = count_trainable_params(equivariant)
print(f"Baseline params:    {p_base:,}")
print(f"Equivariant params: {p_eq:,}")
print(f"Relative gap: {(p_eq - p_base) / p_base:.1%}")

# Practical rule of thumb used in many papers: keep this gap small and report it.


## 3) Proper train / validation / test separation

### Roles of each split

- **Train**: fit model parameters.
- **Validation**: model selection (hyperparameters, architecture variants, stopping).
- **Test**: one-shot final estimate after decisions are fixed.

### Critical rule

Do **not** use test performance to choose hyperparameters, architecture depth, or augmentation strength. That is test leakage.

### Leakage risks in geometric tasks

- Near-duplicate structures across train/test (e.g., conformers, meshes, scenes).
- Shared objects with transformed copies spread across splits.
- Temporal/spatial correlation that makes random splitting optimistic.

### Why random IID splits can hide equivariance benefits

If train and test contain similar transformations, a non-equivariant model can memorize augmentation patterns and appear strong. The real value of equivariance often appears when transformation coverage at test time differs from train time.

Design splits that test the claim:

- Rotation-range split (train on limited angles, test on unseen angles).
- Group-element holdout (train on subset of transformations, test on held-out subset).
- Size/composition split (train on smaller graphs/molecules, test on larger/novel ones).

In [ ]:
# 2) Train/validation/test split design: IID vs structured OOD split
import numpy as np

rng = np.random.default_rng(7)
N = 200
angles = rng.uniform(-180, 180, size=N)  # synthetic rotation metadata (degrees)

# IID random split
iid_idx = rng.permutation(N)
train_iid = angles[iid_idx[:120]]
val_iid = angles[iid_idx[120:160]]
test_iid = angles[iid_idx[160:]]

# Structured split targeting the modeling claim:
# train/val on near-upright angles, test on unseen large rotations
train_struct = angles[(angles >= -30) & (angles <= 30)]
val_struct = angles[(angles > 30) & (angles <= 60)]
test_struct = angles[(angles >= 90) & (angles <= 180)]

print("IID split angle ranges:")
print(f"  train: [{train_iid.min():6.1f}, {train_iid.max():6.1f}] ({len(train_iid)} samples)")
print(f"  val:   [{val_iid.min():6.1f}, {val_iid.max():6.1f}] ({len(val_iid)} samples)")
print(f"  test:  [{test_iid.min():6.1f}, {test_iid.max():6.1f}] ({len(test_iid)} samples)")

print("
Structured split angle ranges (OOD test by design):")
print(f"  train: [{train_struct.min():6.1f}, {train_struct.max():6.1f}] ({len(train_struct)} samples)")
print(f"  val:   [{val_struct.min():6.1f}, {val_struct.max():6.1f}] ({len(val_struct)} samples)")
print(f"  test:  [{test_struct.min():6.1f}, {test_struct.max():6.1f}] ({len(test_struct)} samples)")


## 4) Experiments that expose the value of equivariant architectures

Evaluate at least one regime where symmetry-aware inductive bias should plausibly help:

1. **OOD generalization**
   - Example: unseen rotations, unseen graph sizes, unseen poses.
2. **Sample efficiency**
   - Compare performance vs. training-set size.
3. **Training/optimization efficiency**
   - Compare convergence speed at equal compute budget.
4. **Reduced augmentation dependence**
   - Show whether equivariance reduces heavy augmentation needs.
5. **Equivariance consistency checks**


In [ ]:
# 3) Sample efficiency demo: performance vs training set size
import numpy as np

train_sizes = np.array([50, 100, 200, 400, 800])

# Toy results that mimic a common pattern: equivariant models win more in low-data settings.
baseline_acc = np.array([0.58, 0.64, 0.70, 0.75, 0.79])
equivariant_acc = np.array([0.66, 0.71, 0.76, 0.80, 0.82])

gain = equivariant_acc - baseline_acc
for n, b, e, g in zip(train_sizes, baseline_acc, equivariant_acc, gain):
    print(f"n={n:>3}: baseline={b:.2f}, equivariant={e:.2f}, gain={g:+.2f}")

print(f"
Average gain (all sizes): {gain.mean():+.3f}")
print(f"Average gain (low-data: n<=200): {gain[train_sizes<=200].mean():+.3f}")


In [ ]:
# 4) Multiple seeds + uncertainty template (mean ± std)
import numpy as np

seed_results = {
    "baseline":    np.array([0.742, 0.755, 0.736, 0.748, 0.751]),
    "equivariant": np.array([0.781, 0.776, 0.789, 0.784, 0.780]),
}

for model_name, values in seed_results.items():
    print(f"{model_name:12s}: {values.mean():.3f} ± {values.std(ddof=1):.3f} (n={len(values)} seeds)")

improvement = seed_results["equivariant"] - seed_results["baseline"]
print(f"
Per-seed delta: mean={improvement.mean():+.3f}, std={improvement.std(ddof=1):.3f}")


In [ ]:
# 5) Equivariance consistency check: compare f(g·x) vs rho_out(g)·f(x)
import numpy as np

rng = np.random.default_rng(0)

def R(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def equivariance_error(W, x, theta):
    gx = R(theta) @ x
    left = W @ gx                       # f(g·x)
    right = R(theta) @ (W @ x)          # rho_out(g)·f(x), with rho_out = same rotation
    return np.linalg.norm(left - right) / (np.linalg.norm(right) + 1e-8)

W_equivariant = np.eye(2)               # commutes with all planar rotations
W_noneq = np.array([[1.0, 0.4], [0.1, 0.7]])

errs_eq, errs_non = [], []
for _ in range(200):
    x = rng.normal(size=2)
    theta = rng.uniform(-np.pi, np.pi)
    errs_eq.append(equivariance_error(W_equivariant, x, theta))
    errs_non.append(equivariance_error(W_noneq, x, theta))

print(f"Equivariant map error: mean={np.mean(errs_eq):.4f}")
print(f"Non-equivariant map error: mean={np.mean(errs_non):.4f}")


## 5) Worked mini-case studies (course-relevant)

Below are concrete templates you can adapt. Each includes the task, symmetry, fair comparison, and a regime where equivariance should help.

### Example A — Rotation-equivariant image prediction

- **Prediction problem:** classify image content independent of absolute orientation.
- **Relevant symmetry:** planar rotation.
- **Modeling claim:** rotation equivariance improves OOD rotation robustness and low-data performance.
- **Fair baseline:** matched-parameter CNN trained with same optimizer/schedule/budget.
- **Benefit-revealing experiment:** train on one angle range, test on unseen angle range.


In [ ]:
# Worked example A: make OOD rotation split concrete
import numpy as np

angles_train = np.linspace(-30, 30, 13)
angles_test_ood = np.linspace(90, 180, 10)

print("Train angles:", angles_train)
print("OOD test angles:", angles_test_ood)

# Toy metric table (illustrative, not from full training)
results = {
    "IID test": {"baseline": 0.89, "equivariant": 0.90},
    "OOD rotations": {"baseline": 0.62, "equivariant": 0.79},
}

for split, vals in results.items():
    gap = vals["equivariant"] - vals["baseline"]
    print(f"{split:14s} | baseline={vals['baseline']:.2f} | equivariant={vals['equivariant']:.2f} | gap={gap:+.2f}")


### Example B — Permutation-equivariant graph prediction

- **Prediction problem:** graph-level regression/classification from node features and connectivity.
- **Relevant symmetry:** permutation of node indices.
- **Modeling claim:** permutation-equivariant/invariant models avoid spurious dependence on node ordering.
- **Fair baseline:** non-equivariant MLP baseline with comparable parameters and same training setup.
- **Benefit-revealing experiment:** permute node indices at test time and measure prediction stability.


In [ ]:
# Worked example B: permutation test scaffold
import numpy as np

rng = np.random.default_rng(1)
X = rng.normal(size=(5, 3))  # 5 nodes, 3 features each

# Order-sensitive baseline: flattens in current node order
w_flat = rng.normal(size=X.size)
def baseline_predict(X):
    return float(np.dot(X.reshape(-1), w_flat))

# Permutation-invariant readout: sum over nodes then linear map
w_inv = rng.normal(size=X.shape[1])
def invariant_predict(X):
    return float(np.dot(X.sum(axis=0), w_inv))

p = rng.permutation(len(X))
X_perm = X[p]

b1, b2 = baseline_predict(X), baseline_predict(X_perm)
i1, i2 = invariant_predict(X), invariant_predict(X_perm)

print("Permutation used:", p)
print(f"Baseline (order-sensitive): original={b1:.4f}, permuted={b2:.4f}, |delta|={abs(b1-b2):.4f}")
print(f"Invariant readout:          original={i1:.4f}, permuted={i2:.4f}, |delta|={abs(i1-i2):.4f}")


### Example C — 3D geometric prediction under rigid motions

- **Prediction problem:** 3D prediction (e.g., molecular/property or point-cloud task).
- **Relevant symmetry:** rigid motions in 3D (rotations + translations).
- **Modeling claim:** built-in geometric structure improves consistency and OOD transform robustness.
- **Fair baseline:** non-equivariant 3D network with matched capacity/training budget.
- **Benefit-revealing experiment:** transform stress test and low-data comparison.


In [ ]:
# Worked example C: rigid-motion consistency for scalar vs vector outputs
import numpy as np

rng = np.random.default_rng(2)
P = rng.normal(size=(8, 3))  # toy 3D point cloud

def random_rotation(rng):
    A = rng.normal(size=(3, 3))
    Q, _ = np.linalg.qr(A)
    if np.linalg.det(Q) < 0:
        Q[:, 0] *= -1
    return Q

R = random_rotation(rng)
t = np.array([1.0, -0.5, 0.2])
P_tf = (P @ R.T) + t

# Toy scalar prediction (invariant): mean pairwise distance to centroid
scalar = np.linalg.norm(P - P.mean(axis=0), axis=1).mean()
scalar_tf = np.linalg.norm(P_tf - P_tf.mean(axis=0), axis=1).mean()

# Toy vector prediction (equivariant): principal direction surrogate via centered mean vector
v = (P - P.mean(axis=0)).sum(axis=0)
v_tf = (P_tf - P_tf.mean(axis=0)).sum(axis=0)

print(f"Scalar invariant check | original={scalar:.6f}, transformed={scalar_tf:.6f}, delta={abs(scalar-scalar_tf):.2e}")
print("Vector equivariance check | ||v_tf - R v|| =", np.linalg.norm(v_tf - (R @ v)))


## 6) Ablations and reporting standards

Include ablations that isolate *why* a method works:

- Remove or weaken equivariant components.
- Vary augmentation strength to test whether learned invariance substitutes for built-in equivariance.
- Vary model width/depth under fixed budget.
- Evaluate with multiple random seeds when possible.

Report at minimum:

- Mean ± std over seeds.
- Parameter counts.
- Training budget (epochs/steps, batch size, optimizer/scheduler).
- Runtime or compute discussion when relevant.
- Metrics aligned with claim (e.g., calibration, OOD robustness, equivariance error—not only top-1 accuracy).

### Avoid overclaiming

A tiny improvement (e.g., +0.2%) without uncertainty/context is not strong evidence. Claims should match effect size, variance, and experimental controls.

## 7) Final project checklist (pre-submission)

Use this as a quick audit before submission:

- [ ] I state a **precise hypothesis** linking symmetry prior to expected benefit.
- [ ] I compare against **fair, competitive baselines**.
- [ ] I report or approximately match **parameter counts**.
- [ ] I align **training budget** (optimizer, scheduler, epochs/steps, batch size, early stopping).
- [ ] I maintain a **clean train/val/test split** with no test-driven tuning.
- [ ] I include at least one **symmetry-revealing experiment** (OOD, low-data, reduced augmentation, or consistency test).
- [ ] I include at least one **ablation** isolating the role of equivariance/invariance.
- [ ] I report **metrics aligned with the claim** and include uncertainty (e.g., mean ± std).
- [ ] My conclusions are **proportional to the evidence** and do not overclaim.

---

**Final reminder.** In geometric deep learning, the most convincing project is not the one with the most plots—it is the one where architecture, symmetry assumptions, and empirical evidence form a coherent scientific argument.